<a href="https://colab.research.google.com/github/marcplanas11-alt/insurance-claims-triage-ai/blob/main/Copia_de_claims_langgraph_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LangGraph = tool that controls node order and conditional flow

In [ ]:
!pip install -q langgraph langchain langchain-core


# Claims Agentic Workflow with LangGraph

Goal:
Convert my existing claims workflow into a state-machine-based agentic system.

Insurance use case:
Claims triage with auditability, validation, routing, and human-in-the-loop escalation.

Architecture:
State → Decision Node → Validation Node → Routing Node → Human Review if needed

TypedDict = dictionary with expected fields
Optional = field can be empty / None
List = list of items
Dict = dictionary
Any = flexible value

## My understanding of ClaimState

ClaimState is the digital claim file.

At the beginning it contains:
- claim_id
- claim_text

Each node adds information to the same claim file:
- decision_node adds decision, confidence, and reason
- validation_node adds validation result and errors
- routing_node adds routing and final status

This is useful because the final state shows what happened, why it happened, and whether human review was needed.

ClaimState = evolving digital claim file

In [ ]:
def decision_node(state: ClaimState) -> ClaimState:

    claim_text = state["claim_text"]

    try:
        result = decision_agent_claude(claim_text)

        decision = result.get("decision", "ESCALATE")
        confidence = result.get("confidence", 0.0)
        reason = result.get("reason", "No reason provided")

    except Exception as e:
        decision = "ESCALATE"
        confidence = 0.0
        reason = f"Claude decision failed: {str(e)}"

    state["decision"] = decision
    state["confidence"] = confidence
    state["decision_reason"] = reason

    state["human_review_required"] = (
        decision == "ESCALATE" or confidence < 0.7
    )

    state["audit_log"].append({
        "step": "decision_node",
        "agent": "claude",
        "decision": decision,
        "confidence": confidence,
        "reason": reason,
        "human_review_required": state["human_review_required"]
    })

    return state

In [ ]:
state = {
    "claim_id": "C1",
    "claim_text": "Water damage in kitchen",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "audit_log": []
}

In [ ]:
state = decision_node(state)
print(state)

{'claim_id': 'C1', 'claim_text': 'Water damage in kitchen', 'decision': 'ESCALATE', 'confidence': 0.0, 'decision_reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.0, 'reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'human_review_required': True}]}


In [ ]:
def validation_node(state: ClaimState) -> ClaimState:

    errors = []

    if state["decision"] not in ["APPROVE", "REJECT", "ESCALATE"]:
        errors.append("Invalid decision value")

    if state["confidence"] is None:
        errors.append("Missing confidence score")

    elif state["confidence"] < 0 or state["confidence"] > 1:
        errors.append("Confidence must be between 0 and 1")

    if state["decision_reason"] is None:
        errors.append("Missing decision reason")

    state["is_valid"] = len(errors) == 0
    state["validation_errors"] = errors

    if not state["is_valid"]:
        state["human_review_required"] = True

    state["audit_log"].append({
        "step": "validation_node",
        "is_valid": state["is_valid"],
        "errors": errors
    })

    return state

In [ ]:
state = validation_node(state)
print(state)

{'claim_id': 'C1', 'claim_text': 'Water damage in kitchen', 'decision': 'ESCALATE', 'confidence': 0.0, 'decision_reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'is_valid': True, 'validation_errors': [], 'routing': None, 'final_status': None, 'human_review_required': True, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.0, 'reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'human_review_required': True}, {'step': 'validation_node', 'is_valid': True, 'errors': []}]}


In [ ]:
def routing_node(state: ClaimState) -> ClaimState:

    if not state["is_valid"]:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated due to validation failure"
        state["human_review_required"] = True

    elif state["human_review_required"]:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated for human review"

    elif state["decision"] == "APPROVE" and state["confidence"] >= 0.7:
        routing = "AUTO_PAY"
        final_status = "Approved for automatic payment"

    elif state["decision"] == "REJECT" and state["confidence"] >= 0.7:
        routing = "AUTO_REJECT"
        final_status = "Rejected automatically"

    else:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated due to uncertainty"
        state["human_review_required"] = True

    state["routing"] = routing
    state["final_status"] = final_status

    state["audit_log"].append({
        "step": "routing_node",
        "routing": routing,
        "final_status": final_status,
        "human_review_required": state["human_review_required"]
    })

    return state

In [ ]:
state = routing_node(state)
print(state)

{'claim_id': 'C1', 'claim_text': 'Water damage in kitchen', 'decision': 'ESCALATE', 'confidence': 0.0, 'decision_reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'is_valid': True, 'validation_errors': [], 'routing': 'MANUAL_REVIEW', 'final_status': 'Escalated for human review', 'human_review_required': True, 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.0, 'reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'human_review_required': True}, {'step': 'validation_node', 'is_valid': True, 'errors': []}, {'step': 'routing_node', 'routing': 'MANUAL_REVIEW', 'final_status': 'Escalated for human review', 'human_review_required': True}]}


In [ ]:
def decide_next_step(state: ClaimState) -> str:

    if state["human_review_required"]:
        return "human_review"

    return "validation"

In [ ]:
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, START, END
import json

class ClaimState(TypedDict):
    claim_id: str
    claim_text: str

    decision: Optional[str]
    confidence: Optional[float]
    decision_reason: Optional[str]

    is_valid: Optional[bool]
    validation_errors: Optional[List[str]]

    routing: Optional[str]
    final_status: Optional[str]

    human_review_required: bool
    human_review_reason: Optional[str]
    human_reviewer: Optional[str]
    human_review_status: Optional[str]

    audit_log: List[Dict[str, Any]]

# --- Helper functions moved here to resolve NameErrors ---
def safe_json_parse(raw_text: str) -> dict:

    cleaned = raw_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()

    elif cleaned.startswith("```"):
        cleaned = cleaned.replace("```", "").strip()

    return json.loads(cleaned)

def decision_agent_claude(claim_text: str) -> dict:
    # client object will be used implicitly from global scope, assuming it's defined elsewhere

    prompt = f"""
You are an insurance claims triage assistant.

Analyze the claim text and return ONLY valid JSON.

Allowed decisions:
- APPROVE
- REJECT
- ESCALATE

Rules:
- Use ESCALATE when information is unclear.
- Use ESCALATE when confidence is below 0.7.
- Do not invent facts.
- Do not assume coverage exists.
- Return JSON only.

Claim text:
{claim_text}

Required JSON format:
{{
  "decision": "APPROVE | REJECT | ESCALATE",
  "confidence": 0.0,
  "reason": "short explanation"
}}
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = response.content[0].text

    # Use safe_json_parse to handle markdown code fences
    return safe_json_parse(raw_text)
# --- End of helper functions ---

# --- Function definitions from other cells, moved here to resolve NameErrors ---
def decision_node(state: ClaimState) -> ClaimState:

    claim_text = state["claim_text"]

    try:
        result = decision_agent_claude(claim_text)

        decision = result.get("decision", "ESCALATE")
        confidence = result.get("confidence", 0.0)
        reason = result.get("reason", "No reason provided")

    except Exception as e:
        decision = "ESCALATE"
        confidence = 0.0
        reason = f"Claude decision failed: {str(e)}"

    state["decision"] = decision
    state["confidence"] = confidence
    state["decision_reason"] = reason

    state["human_review_required"] = (
        decision == "ESCALATE" or confidence < 0.7
    )

    state["audit_log"].append({
        "step": "decision_node",
        "agent": "claude",
        "decision": decision,
        "confidence": confidence,
        "reason": reason,
        "human_review_required": state["human_review_required"]
    })

    return state

def validation_node(state: ClaimState) -> ClaimState:

    errors = []

    if state["decision"] not in ["APPROVE", "REJECT", "ESCALATE"]:
        errors.append("Invalid decision value")

    if state["confidence"] is None:
        errors.append("Missing confidence score")

    elif state["confidence"] < 0 or state["confidence"] > 1:
        errors.append("Confidence must be between 0 and 1")

    if state["decision_reason"] is None:
        errors.append("Missing decision reason")

    state["is_valid"] = len(errors) == 0
    state["validation_errors"] = errors

    if not state["is_valid"]:
        state["human_review_required"] = True

    state["audit_log"].append({
        "step": "validation_node",
        "is_valid": state["is_valid"],
        "errors": errors
    })

    return state

def routing_node(state: ClaimState) -> ClaimState:

    if not state["is_valid"]:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated due to validation failure"
        state["human_review_required"] = True

    elif state["human_review_required"]:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated for human review"

    elif state["decision"] == "APPROVE" and state["confidence"] >= 0.7:
        routing = "AUTO_PAY"
        final_status = "Approved for automatic payment"

    elif state["decision"] == "REJECT" and state["confidence"] >= 0.7:
        routing = "AUTO_REJECT"
        final_status = "Rejected automatically"

    else:
        routing = "MANUAL_REVIEW"
        final_status = "Escalated due to uncertainty"
        state["human_review_required"] = True

    state["routing"] = routing
    state["final_status"] = final_status

    state["audit_log"].append({
        "step": "routing_node",
        "routing": routing,
        "final_status": final_status,
        "human_review_required": state["human_review_required"]
    })

    return state

def decide_next_step(state: ClaimState) -> str:

    if state["human_review_required"]:
        return "human_review"

    return "validation"

# Placeholder for human_review_node, as it was used but not defined in context
def human_review_node(state: ClaimState) -> ClaimState:
    # In a real scenario, this would involve a human interaction
    state["human_review_status"] = "PENDING"
    state["audit_log"].append({
        "step": "human_review_node",
        "status": "PENDING"
    })
    return state

# --- End of function definitions ---

workflow = StateGraph(ClaimState)

workflow.add_node("decision", decision_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("validation", validation_node)
workflow.add_node("routing", routing_node)

workflow.add_edge(START, "decision")

workflow.add_conditional_edges(
    "decision",
    decide_next_step,
    {
        "human_review": "human_review",
        "validation": "validation"
    }
)

workflow.add_edge("human_review", END)
workflow.add_edge("validation", "routing")
workflow.add_edge("routing", END)

claims_graph = workflow.compile()

In [ ]:
initial_state = {
    "claim_id": "C2",
    "claim_text": "Water damage in bathroom after pipe leak",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)

print(final_state)

{'claim_id': 'C2', 'claim_text': 'Water damage in bathroom after pipe leak', 'decision': 'ESCALATE', 'confidence': 0.0, 'decision_reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.0, 'reason': "Claude decision failed: name 'decision_agent_claude' is not defined", 'human_review_required': True}, {'step': 'human_review_node', 'status': 'PENDING'}]}


## LangGraph Claims Workflow Summary

I built a state-machine claims workflow using LangGraph.

The workflow includes:
- decision_node: makes an initial claims decision
- validation_node: checks decision quality and required fields
- routing_node: applies deterministic claims routing

Governance controls:
- JSON-like structured state
- confidence threshold
- validation layer
- fallback to manual review
- audit log for every node

Insurance relevance:
This mirrors a controlled MGA claims triage process where AI supports decisions but Python routing and human-in-the-loop controls govern the final outcome.

In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

Enter your Anthropic API key: ··········


In [ ]:
from anthropic import Anthropic
import json

client = Anthropic()

In [ ]:
def decision_agent_claude(claim_text: str) -> dict:

    prompt = f"""
You are an insurance claims triage assistant.

Analyze the claim text and return ONLY valid JSON.

Allowed decisions:
- APPROVE
- REJECT
- ESCALATE

Rules:
- Use ESCALATE when information is unclear.
- Use ESCALATE when confidence is below 0.7.
- Do not invent facts.
- Do not assume coverage exists.
- Return JSON only.

Claim text:
{claim_text}

Required JSON format:
{{
  "decision": "APPROVE | REJECT | ESCALATE",
  "confidence": 0.0,
  "reason": "short explanation"
}}
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = response.content[0].text
    print("RAW CLAUDE OUTPUT:")
    print(raw_text)

    return json.loads(raw_text)

In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)

claude-opus-4-7
claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514


In [ ]:
def safe_json_parse(raw_text: str) -> dict:

    cleaned = raw_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()

    elif cleaned.startswith("```"):
        cleaned = cleaned.replace("```", "").strip()

    return json.loads(cleaned)

In [ ]:
def decision_agent_claude(claim_text: str) -> dict:

    prompt = f"""
You are an insurance claims triage assistant.

Analyze the claim text and return ONLY valid JSON.

Allowed decisions:
- APPROVE
- REJECT
- ESCALATE

Rules:
- Use ESCALATE when information is unclear.
- Use ESCALATE when confidence is below 0.7.
- Do not invent facts.
- Do not assume coverage exists.
- Return JSON only.

Claim text:
{claim_text}

Required JSON format:
{{
  "decision": "APPROVE | REJECT | ESCALATE",
  "confidence": 0.0,
  "reason": "short explanation"
}}
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_text = response.content[0].text
    print("RAW CLAUDE OUTPUT:")
    print(raw_text)

    # Fix: Use safe_json_parse to handle markdown code fences
    return safe_json_parse(raw_text)

test_result = decision_agent_claude("Water damage in kitchen after pipe burst")
print(test_result)

RAW CLAUDE OUTPUT:
```json
{
  "decision": "ESCALATE",
  "confidence": 0.5,
  "reason": "Insufficient information to determine coverage. Need policy details, cause verification, maintenance history, and whether sudden/accidental damage is covered under the specific policy."
}
```
{'decision': 'ESCALATE', 'confidence': 0.5, 'reason': 'Insufficient information to determine coverage. Need policy details, cause verification, maintenance history, and whether sudden/accidental damage is covered under the specific policy.'}


In [ ]:
initial_state = {
    "claim_id": "C4",
    "claim_text": "Water damage in kitchen after pipe burst",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)

print(final_state)

RAW CLAUDE OUTPUT:
```json
{
  "decision": "ESCALATE",
  "confidence": 0.5,
  "reason": "Insufficient information to determine coverage. Need policy details, cause verification, maintenance history, and whether damage is from sudden/accidental burst versus negligence or lack of maintenance."
}
```
{'claim_id': 'C4', 'claim_text': 'Water damage in kitchen after pipe burst', 'decision': 'ESCALATE', 'confidence': 0.5, 'decision_reason': 'Insufficient information to determine coverage. Need policy details, cause verification, maintenance history, and whether damage is from sudden/accidental burst versus negligence or lack of maintenance.', 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.5, 'reason': 'Insufficient information to determine coverage. Need policy details, cause verification, maint

In [ ]:
def decide_next_step(state: ClaimState) -> str:

    if state["human_review_required"]:
        return "end"

    return "validation"

In [ ]:
initial_state = {
    "claim_id": "C5",
    "claim_text": "Customer reports unusual situation with unclear details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

RAW CLAUDE OUTPUT:
```json
{
  "decision": "ESCALATE",
  "confidence": 0.1,
  "reason": "Claim details are unclear and insufficient. Cannot determine coverage applicability or claim validity without additional information."
}
```
{'claim_id': 'C5', 'claim_text': 'Customer reports unusual situation with unclear details', 'decision': 'ESCALATE', 'confidence': 0.1, 'decision_reason': 'Claim details are unclear and insufficient. Cannot determine coverage applicability or claim validity without additional information.', 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.1, 'reason': 'Claim details are unclear and insufficient. Cannot determine coverage applicability or claim validity without additional information.', 'human_review_required': True}, {'step': 'human_review_node', 'status': 'PENDING'

In [ ]:
initial_state = {
    "claim_id": "C6",
    "claim_text": "Minor scratch on insured vehicle reported with full details",

    "decision": None,
    "confidence": None,
    "decision_reason": None,

    "is_valid": None,
    "validation_errors": None,

    "routing": None,
    "final_status": None,

    "human_review_required": False,
    "audit_log": []
}

final_state = claims_graph.invoke(initial_state)
print(final_state)

RAW CLAUDE OUTPUT:
```json
{
  "decision": "ESCALATE",
  "confidence": 0.5,
  "reason": "Insufficient information to make determination. Need details on: policy coverage type, deductible amount, damage assessment, repair estimate, and whether comprehensive/collision coverage applies to this claim."
}
```
{'claim_id': 'C6', 'claim_text': 'Minor scratch on insured vehicle reported with full details', 'decision': 'ESCALATE', 'confidence': 0.5, 'decision_reason': 'Insufficient information to make determination. Need details on: policy coverage type, deductible amount, damage assessment, repair estimate, and whether comprehensive/collision coverage applies to this claim.', 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.5, 'reason': 'Insufficient information to make determination. Need details 

In [ ]:
def human_review_node(state: ClaimState) -> ClaimState:

    review_reason = state.get("decision_reason", "Manual review required")

    state["human_review_status"] = "PENDING"
    state["routing"] = "MANUAL_REVIEW"
    state["final_status"] = "Pending human claims review"

    state["audit_log"].append({
        "step": "human_review_node",
        "review_required": True,
        "review_reason": review_reason,
        "assigned_to": "claims_adjuster",
        "status": "PENDING"
    })

    return state

In [ ]:
print(final_state)

{'claim_id': 'C6', 'claim_text': 'Minor scratch on insured vehicle reported with full details', 'decision': 'ESCALATE', 'confidence': 0.5, 'decision_reason': 'Insufficient information to make determination. Need details on: policy coverage type, deductible amount, damage assessment, repair estimate, and whether comprehensive/collision coverage applies to this claim.', 'is_valid': None, 'validation_errors': None, 'routing': None, 'final_status': None, 'human_review_required': True, 'human_review_status': 'PENDING', 'audit_log': [{'step': 'decision_node', 'agent': 'claude', 'decision': 'ESCALATE', 'confidence': 0.5, 'reason': 'Insufficient information to make determination. Need details on: policy coverage type, deductible amount, damage assessment, repair estimate, and whether comprehensive/collision coverage applies to this claim.', 'human_review_required': True}, {'step': 'human_review_node', 'status': 'PENDING'}]}
